# 08 — Knowledge Distillation Basics

> Companion to **Week 11**, Part 6 of the lecture.

## The idea

A big **teacher** model trains a small **student** model. Instead of learning
from raw labels alone, the student learns from the teacher's *soft predictions*
("90% cat, 8% lynx, 2% dog"). This extra information is rich, and lets the
student get away with much fewer parameters.

In this notebook we will:

1. Train a **big teacher** on the full training set.
2. Train a small **student** on a *small* subset using only hard labels.
3. Train another small **student** on the same small subset, but using the
   teacher's soft predictions.
4. Compare the three.


## Step 1 — Set up data and model classes


In [ ]:
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, TensorDataset

torch.manual_seed(42)

digits = load_digits()
X = torch.tensor(digits.data / 16.0, dtype=torch.float32)
y = torch.tensor(digits.target, dtype=torch.long)

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# The students get to see only a SMALL slice of training data.
X_train_small = X_train_full[:200]
y_train_small = y_train_full[:200]

full_loader  = DataLoader(TensorDataset(X_train_full, y_train_full), batch_size=64, shuffle=True)
small_loader = DataLoader(TensorDataset(X_train_small, y_train_small), batch_size=32, shuffle=True)
test_loader  = DataLoader(TensorDataset(X_test, y_test), batch_size=256)


class Teacher(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.net(x)


class Student(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 10),
        )

    def forward(self, x):
        return self.net(x)


def evaluate(model):
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in test_loader:
            preds = model(xb).argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += len(yb)
    return correct / total


def count_params(m):
    return sum(p.numel() for p in m.parameters())


teacher           = Teacher()
student_hard      = Student()
student_distilled = Student()

print("Teacher parameters:", count_params(teacher))
print("Student parameters:", count_params(student_hard))


## Step 2 — Train the big teacher on the full dataset


In [ ]:
teacher_optimizer = torch.optim.Adam(teacher.parameters(), lr=0.01)
loss_fn = nn.CrossEntropyLoss()

for epoch in range(10):
    teacher.train()
    for xb, yb in full_loader:
        loss = loss_fn(teacher(xb), yb)
        teacher_optimizer.zero_grad()
        loss.backward()
        teacher_optimizer.step()

teacher_acc = evaluate(teacher)
print(f"Teacher accuracy: {teacher_acc:.3f}")


## Step 3 — Train a student WITHOUT distillation

We give it only the **small** training set and hard labels.


In [ ]:
hard_optimizer = torch.optim.Adam(student_hard.parameters(), lr=0.01)

for epoch in range(15):
    student_hard.train()
    for xb, yb in small_loader:
        loss = loss_fn(student_hard(xb), yb)
        hard_optimizer.zero_grad()
        loss.backward()
        hard_optimizer.step()

student_hard_acc = evaluate(student_hard)
print(f"Hard-label student accuracy: {student_hard_acc:.3f}")


## Step 4 — Train a second student WITH distillation

Same architecture, same small training set. The only difference: we ALSO
match the teacher's *soft* predictions.

The two ingredients of distillation:

- **temperature** — softens the teacher's probabilities so the small numbers
  ("8% lynx, 2% dog") become visible to the student.
- **alpha** — how much weight to give the soft loss vs the hard loss.


In [ ]:
distill_optimizer = torch.optim.Adam(student_distilled.parameters(), lr=0.01)
temperature = 3.0
alpha = 0.5

teacher.eval()
for epoch in range(15):
    student_distilled.train()
    for xb, yb in small_loader:
        with torch.no_grad():
            teacher_logits = teacher(xb)

        student_logits = student_distilled(xb)

        soft_loss = F.kl_div(
            F.log_softmax(student_logits / temperature, dim=1),
            F.softmax(teacher_logits / temperature, dim=1),
            reduction="batchmean",
        ) * (temperature ** 2)

        hard_loss = loss_fn(student_logits, yb)

        loss = alpha * soft_loss + (1 - alpha) * hard_loss

        distill_optimizer.zero_grad()
        loss.backward()
        distill_optimizer.step()

student_distilled_acc = evaluate(student_distilled)
print(f"Distilled student accuracy: {student_distilled_acc:.3f}")


## Step 5 — Compare all three


In [ ]:
pd.DataFrame(
    [
        {"model": "teacher",            "params": count_params(teacher),           "accuracy": teacher_acc},
        {"model": "student_hard",       "params": count_params(student_hard),      "accuracy": student_hard_acc},
        {"model": "student_distilled",  "params": count_params(student_distilled), "accuracy": student_distilled_acc},
    ]
)


### What you should see

- The teacher is the most accurate (it had the most data and the most parameters).
- The hard-label student is the **least** accurate.
- The distilled student lands in between, often much closer to the teacher
  than to the hard-label student.

Two punchlines:

1. The **same small architecture** can do dramatically better when trained
   from a teacher's soft labels.
2. The student in production is **as small as the hard-label student** — same
   memory, same latency. You only paid the teacher cost during training.

## 🧪 Your turn

Set `temperature = 1.0` and re-run Step 4. What happens to the distilled
student? Why? (Hint: what does temperature do to the teacher's probabilities?)
